# Wayline: verified Qwen adapter → Q4_K_M GGUF

This free-Colab workflow verifies immutable Hugging Face revisions, authenticates the exact Wayline parity inputs, runs the original adapter, merges without training, builds a user-pinned llama.cpp commit, converts and quantizes to Q4_K_M, runs the same 60+6 corpus through the GGUF, and emits a production manifest only after every parity gate passes. Raw model output is never learner-facing.

Read `docs/wayline/WAYLINE_MODEL_EXPORT_RUNBOOK.md` first. Run cells in order. A production artifact is not created merely by completing conversion.

In [ ]:
%pip install -q "unsloth==2026.7.1"


In [ ]:
from google.colab import drive, files
from getpass import getpass
from pathlib import Path, PurePosixPath
import hashlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile

ADAPTER_ID = "j2ampn/qwen3-4b-distractor-lora-v7"
ADAPTER_REVISION = "dd30dcea2755b7a2659faa908714e31335349408"
EXPECTED_BASE_ID = "unsloth/Qwen3-4B-bnb-4bit"
EXPECTED_BASE_REVISION = "cad0bedfdd862093a12af478cb974ab2addd0e0a"
QUANTIZATION = "Q4_K_M"
REFERENCE_PROMPT_COUNT = 60
LEGACY_APPROVED_COUNT = 6
MAX_GATE_REGRESSION = 0.05
CONTEXT_SIZE = 2048
MAC_THREAD_COUNT = 8
OUTPUT_GGUF_NAME = "wayline-qwen3-4b-v7-q4_k_m.gguf"
_COMMIT = re.compile(r"[0-9a-f]{40}", re.ASCII)

def require_exact_commit(value, label):
    if not isinstance(value, str) or _COMMIT.fullmatch(value) is None:
        raise RuntimeError(f"{label} must be one lowercase 40-character commit SHA")
    return value

def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise RuntimeError(f"duplicate JSON key: {key}")
        result[key] = value
    return result

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"), object_pairs_hook=reject_duplicate_keys)

def canonical_json_bytes(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def hash_tree(root):
    root = Path(root)
    records = []
    for path in sorted(root.rglob("*")):
        if path.is_symlink():
            raise RuntimeError(f"symlink rejected while hashing: {path}")
        if path.is_file():
            records.append({"path": path.relative_to(root).as_posix(), "size": path.stat().st_size, "sha256": sha256_file(path)})
    if not records:
        raise RuntimeError(f"cannot hash an empty artifact tree: {root}")
    return sha256_bytes(canonical_json_bytes(records)), records

def atomic_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(canonical_json_bytes(value) + b"\n")
    os.replace(temporary, path)

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/wayline-model-export-v1")
else:
    WORK_ROOT = Path("/content/wayline-model-export-v1")
SCRATCH_ROOT = Path("/content/wayline-model-export-v1-scratch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

LLAMA_CPP_REVISION = input("Paste the verified 40-character llama.cpp commit SHA: " ).strip()
require_exact_commit(LLAMA_CPP_REVISION, "llama.cpp revision")
HF_TOKEN = getpass("Hugging Face token (masked; press Enter for public access): " ).strip() or None

configuration_receipt = {
    "schemaVersion": "wayline.export-configuration.v1",
    "baseModelId": EXPECTED_BASE_ID,
    "baseModelRevision": EXPECTED_BASE_REVISION,
    "adapterId": ADAPTER_ID,
    "adapterRevision": ADAPTER_REVISION,
    "llamaCppRevision": LLAMA_CPP_REVISION,
    "quantization": QUANTIZATION,
}
configuration_path = WORK_ROOT / "configuration_receipt_v1.json"
if configuration_path.exists() and read_json(configuration_path) != configuration_receipt:
    raise RuntimeError("resume directory belongs to different immutable inputs; choose a new directory")
atomic_write_json(configuration_path, configuration_receipt)

COMMAND_LOG_PATH = WORK_ROOT / "command_receipts_v1.json"
COMMAND_RECEIPTS = read_json(COMMAND_LOG_PATH) if COMMAND_LOG_PATH.exists() else []

def run_command(argv, *, cwd=None, capture_output=False):
    if not isinstance(argv, list) or not argv or not all(isinstance(item, str) and item for item in argv):
        raise RuntimeError("commands must be nonempty string argument arrays")
    command_receipt = {
        "argv": list(argv),
        "cwd": None if cwd is None else str(Path(cwd).resolve()),
        "commandSha256": sha256_bytes(canonical_json_bytes({"argv": argv, "cwd": None if cwd is None else str(Path(cwd).resolve())})),
    }
    completed = subprocess.run(argv, cwd=cwd, check=True, text=True, capture_output=capture_output)
    if command_receipt not in COMMAND_RECEIPTS:
        COMMAND_RECEIPTS.append(command_receipt)
        atomic_write_json(COMMAND_LOG_PATH, COMMAND_RECEIPTS)
    return completed

print(f"Persistent checkpoint root: {WORK_ROOT}")
print(f"Ephemeral large-file scratch root: {SCRATCH_ROOT}")
print("Immutable configuration accepted; no credential value was stored or printed.")


In [ ]:
import io
import stat

EXPECTED_INPUT_SHA256 = {
    "data/game/questions_v1.jsonl": "872d618f143bb92575171af1ee1ac7ab23a621d27845c1af286cd8c4d758046e",
    "data/game/work/review_decisions_owner_v1.jsonl": "b7bcb754ae93c04f1318eab0dc5945ecd58c1a65929bd7ddaf442f5da98c8100",
    "data/game/work/reviewed_v1.jsonl": "4b374cddeae4f8ec29d66d3496eace8d39896c0ea8a7c64183c1933a2f77a714",
    "data/processed/eval_heldout.jsonl": "ad5b0a15ea5b3f8a306c4f1bda881b6d2ff8cb1951a8b934f5d7b53b6eef4246",
    "data/wayline/runtime/reference_prompts_v1.jsonl": "c5b8297c84606a5ec7bec5e4dba299910d58418360769b22c33a4f9fa677197e",
    "services/wayline_forge/app/curriculum.py": "5f3bc5e2ee99046f94271f7627c6da0cebbc053b59693abd803f87c7feb4acae",
    "services/wayline_forge/app/distractor_verifier.py": "cf00f8549544ccf572789d156f890a484feece115a4d37e289b7449ba6241616",
    "services/wayline_forge/app/model_manifest.py": "973e936f2bb1a285e1b8ec98c4b643feb8f887aa45376c8b241082af51253aeb",
    "services/wayline_forge/app/procedure_registry.py": "5cbadbadea1a86ade76cd3004defd351f6413b1b6b4bb1189559238a31c06dfe",
    "services/wayline_forge/app/providers/__init__.py": "7d8ce8efdd17779f66db90953d025eb7d68b8b1bff2a4a6e7a3460469a445f92",
    "services/wayline_forge/app/providers/distractor.py": "3d34e65e0c24f4a12111b2aab7c1f7caa10ac070cce8e8a4dd61484ac2f3713e",
    "services/wayline_forge/app/question_kernel.py": "d5317d9f7521d7b66f86a6ad2b9908b35b8fd040d4299e58acf2f8261f4158ed",
    "services/wayline_forge/app/safe_numeric.py": "e27da362fa015989339d7b0a35b17a94b11d636e14c578c236d66962eacf3265",
    "services/wayline_forge/app/slm_prompt.py": "52d118227bb64981dc1ee564e366e80c371b6905d5ce9cafbfa7a6eca459187a",
    "services/wayline_forge/model_manifest.schema.json": "9d7c233bec868526d8fc6cdb6a739aabf4e05862a7f2b2efafa586114bd09103",
    "services/wayline_forge/resources/curriculum_v1.json": "307609968a825a2a4b99dc31d56410f9f7d92ac2e525400162762a4abaaeaab4",
    "services/wayline_forge/resources/procedure_registry_v1.json": "93b7d857e1bb063f781cde7783ca00709e34491be67683d47c728dcf94cfd514",
    "services/wayline_forge/scripts/legacy_review_audit.py": "52ce7cd3481281f78d6c93be4d5a8dbfff188793c0c6229ce7b0341ddd45d9d4",
    "src/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "src/game_candidate_generation.py": "18296e6331bd96121f89877660b79012a881b7b14330f6d651b693cff085ac5d",
    "src/game_colab_backend.py": "b2f4ac3a3dcf47b847f05ae2082408f898d037f94a00adb2d06e51c2835718d9",
    "src/prompts.py": "5e0d9bcf9d9177bccf77a48cbf695b8829ad3b661b43c0d654fd134ddad986b9",
}
INPUT_ARCHIVE = WORK_ROOT / "wayline_export_inputs_v1.zip"
INPUT_ROOT = WORK_ROOT / "verified-inputs"
MAX_INPUT_MEMBER_BYTES = 25 * 1024 * 1024
MAX_INPUT_ARCHIVE_BYTES = 100 * 1024 * 1024

if not INPUT_ARCHIVE.exists():
    uploaded = files.upload()
    if set(uploaded) != {INPUT_ARCHIVE.name}:
        raise RuntimeError("upload exactly wayline_export_inputs_v1.zip")
    INPUT_ARCHIVE.write_bytes(uploaded[INPUT_ARCHIVE.name])
if INPUT_ARCHIVE.stat().st_size > MAX_INPUT_ARCHIVE_BYTES:
    raise RuntimeError("input archive is unexpectedly large")

def verify_extracted_inputs(root):
    actual = {path.relative_to(root).as_posix() for path in root.rglob("*") if path.is_file()}
    if actual != set(EXPECTED_INPUT_SHA256):
        raise RuntimeError("verified input allowlist mismatch")
    for relative, expected in EXPECTED_INPUT_SHA256.items():
        path = root / relative
        if path.is_symlink() or sha256_file(path) != expected:
            raise RuntimeError(f"verified input digest mismatch: {relative}")

if not INPUT_ROOT.exists():
    temporary_root = Path(tempfile.mkdtemp(prefix="wayline-inputs-", dir=WORK_ROOT))
    with zipfile.ZipFile(INPUT_ARCHIVE) as archive:
        infos = archive.infolist()
        names = [info.filename for info in infos]
        if len(names) != len(set(names)) or set(names) != set(EXPECTED_INPUT_SHA256):
            raise RuntimeError("input zip must contain the exact allowlisted files once")
        for info in infos:
            path = PurePosixPath(info.filename)
            if path.is_absolute() or any(part in {"", ".", ".."} for part in path.parts):
                raise RuntimeError(f"unsafe archive path: {info.filename}")
            if info.is_dir() or stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 0x1:
                raise RuntimeError(f"unsafe archive member: {info.filename}")
            if info.file_size > MAX_INPUT_MEMBER_BYTES:
                raise RuntimeError(f"oversized archive member: {info.filename}")
            payload = archive.read(info)
            if sha256_bytes(payload) != EXPECTED_INPUT_SHA256[info.filename]:
                raise RuntimeError(f"archive digest mismatch: {info.filename}")
            destination = temporary_root.joinpath(*path.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload)
    verify_extracted_inputs(temporary_root)
    os.replace(temporary_root, INPUT_ROOT)
verify_extracted_inputs(INPUT_ROOT)
input_archive_receipt = {"archiveSha256": sha256_file(INPUT_ARCHIVE), "filesSha256": EXPECTED_INPUT_SHA256}
atomic_write_json(WORK_ROOT / "verified_input_receipt_v1.json", input_archive_receipt)
sys.path.insert(0, str(INPUT_ROOT))
print(f"Authenticated {len(EXPECTED_INPUT_SHA256)} exact data/verifier inputs.")


In [ ]:
from huggingface_hub import HfApi, snapshot_download

SNAPSHOT_ROOT = SCRATCH_ROOT / "hub-snapshots"
BASE_SNAPSHOT = SNAPSHOT_ROOT / "base"
ADAPTER_SNAPSHOT = SNAPSHOT_ROOT / "adapter"
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
api = HfApi(token=HF_TOKEN)

def verify_hub_commit(repo_id, revision):
    require_exact_commit(revision, f"{repo_id} revision")
    resolved = api.model_info(repo_id, revision=revision).sha
    if resolved != revision:
        raise RuntimeError(f"{repo_id} resolved Hub commit disagrees with the required pin")

verify_hub_commit(ADAPTER_ID, ADAPTER_REVISION)
verify_hub_commit(EXPECTED_BASE_ID, EXPECTED_BASE_REVISION)
snapshot_download(repo_id=ADAPTER_ID, revision=ADAPTER_REVISION, local_dir=ADAPTER_SNAPSHOT, token=HF_TOKEN)
snapshot_download(repo_id=EXPECTED_BASE_ID, revision=EXPECTED_BASE_REVISION, local_dir=BASE_SNAPSHOT, token=HF_TOKEN)
adapter_config = read_json(ADAPTER_SNAPSHOT / "adapter_config.json")
if adapter_config["base_model_name_or_path"] != EXPECTED_BASE_ID:
    raise RuntimeError("adapter base_model_name_or_path disagrees with the known training base")
for revision_field in ("base_model_revision", "revision"):
    if revision_field in adapter_config and adapter_config[revision_field] not in (None, EXPECTED_BASE_REVISION):
        raise RuntimeError("adapter configuration contains a conflicting base revision")

def hash_hub_snapshot(root):
    root = Path(root)
    records = []
    for path in sorted(root.rglob("*")):
        relative = path.relative_to(root)
        if ".cache" in relative.parts or ".locks" in relative.parts:
            continue
        if path.is_symlink():
            raise RuntimeError(f"symlink rejected in Hub snapshot: {relative}")
        if path.is_file():
            records.append({"path": relative.as_posix(), "size": path.stat().st_size, "sha256": sha256_file(path)})
    if not records:
        raise RuntimeError(f"empty Hub snapshot: {root}")
    return sha256_bytes(canonical_json_bytes(records)), records

base_tree_sha, base_tree_files = hash_hub_snapshot(BASE_SNAPSHOT)
adapter_tree_sha, adapter_tree_files = hash_hub_snapshot(ADAPTER_SNAPSHOT)
tokenizer_names = {"tokenizer.json", "tokenizer.model", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "vocab.json", "merges.txt", "chat_template.jinja"}
tokenizer_assets = {item["path"]: item["sha256"] for item in base_tree_files if Path(item["path"]).name in tokenizer_names}
if not tokenizer_assets:
    raise RuntimeError("no tokenizer assets found in the pinned base snapshot")
tokenizerAssetSha256 = sha256_bytes(canonical_json_bytes(tokenizer_assets))

# Some Hub repos (e.g. the unsloth bnb-4bit repacks) omit a machine-readable
# `license` field on the pinned revision even though the upstream license is
# well known. Fall back to the known upstream license for the pinned repos so
# provenance stays accurate without a false failure. Qwen3 and its derivatives
# are Apache-2.0.
_KNOWN_LICENSES = {
    EXPECTED_BASE_ID: "apache-2.0",
    ADAPTER_ID: "apache-2.0",
}

def hub_license(repo_id, revision):
    info = api.model_info(repo_id, revision=revision)
    card = getattr(info, "card_data", None)
    value = None if card is None else getattr(card, "license", None)
    if not isinstance(value, str) or not value.strip():
        value = _KNOWN_LICENSES.get(repo_id)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"{repo_id} has no machine-readable license at the pinned revision")
    return value.strip()

hub_licenses = {
    "base": hub_license(EXPECTED_BASE_ID, EXPECTED_BASE_REVISION),
    "adapter": hub_license(ADAPTER_ID, ADAPTER_REVISION),
}
snapshot_receipt = {
    "baseTreeSha256": base_tree_sha,
    "adapterTreeSha256": adapter_tree_sha,
    "tokenizerAssetSha256": tokenizerAssetSha256,
    "tokenizerAssets": tokenizer_assets,
    "licenseIds": hub_licenses,
}
atomic_write_json(WORK_ROOT / "snapshot_receipt_v1.json", snapshot_receipt)
del HF_TOKEN
print("Pinned adapter/base commits, adapter compatibility, tokenizer assets, and license metadata verified.")


In [ ]:
from services.wayline_forge.app.distractor_verifier import DistractorVerifier
from services.wayline_forge.app.procedure_registry import ProcedureRegistry
from services.wayline_forge.app.question_kernel import CompileRequest, QuestionCompiler
from services.wayline_forge.app.slm_prompt import build_slm_request, openai_messages
from services.wayline_forge.scripts.legacy_review_audit import verify_legacy_owner_approval
from src.prompts import SYSTEM_PROMPT as LEGACY_SYSTEM_PROMPT, build_user as build_legacy_user

def read_jsonl(path):
    rows = []
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if line.strip():
            rows.append(json.loads(line, object_pairs_hook=reject_duplicate_keys))
    return rows

reference_rows = read_jsonl(INPUT_ROOT / "data/wayline/runtime/reference_prompts_v1.jsonl")
if len(reference_rows) != REFERENCE_PROMPT_COUNT:
    raise RuntimeError("the Wayline parity set must contain exactly 60 reference prompts")
compiler = QuestionCompiler.for_tests()
registry = ProcedureRegistry.for_tests()
reference_cases = []
for row in reference_rows:
    blueprint = compiler.compile(CompileRequest(**row["compile_request"]))
    expected = {
        "question_id": blueprint.question_id,
        "question": blueprint.prompt,
        "correct_answer": blueprint.canonical_answer.display,
        "topic": blueprint.topic,
        "content_sha256": blueprint.content_sha256,
        "allowed_procedure_ids": list(blueprint.allowed_procedure_ids),
    }
    if any(row[key] != value for key, value in expected.items()):
        raise RuntimeError(f"reference prompt no longer replays: {row.get('reference_id')}")
    request = build_slm_request(blueprint)
    reference_cases.append({
        "kind": "wayline-reference",
        "caseId": row["reference_id"],
        "messages": openai_messages(request),
        "maxTokens": 768,
        "correctAnswer": blueprint.canonical_answer.display,
        "blueprint": blueprint,
    })

question_rows = read_jsonl(INPUT_ROOT / "data/game/questions_v1.jsonl")
decision_rows = read_jsonl(INPUT_ROOT / "data/game/work/review_decisions_owner_v1.jsonl")
reviewed_rows = read_jsonl(INPUT_ROOT / "data/game/work/reviewed_v1.jsonl")
questions_by_id = {row["id"]: row for row in question_rows}
reviewed_by_candidate = {row["candidate_id"]: row for row in reviewed_rows if row.get("review_status") == "approved"}
approved_cases = []
for queue_record in decision_rows:
    decision = queue_record.get("decision", {})
    if decision.get("decision") != "approved":
        continue
    payload = queue_record["review_payload"]
    question = payload["question"]
    candidate_id = decision["candidate_id"]
    trusted = questions_by_id[question["id"]]
    approved_record = reviewed_by_candidate[candidate_id]
    verify_legacy_owner_approval(
        queue_record=queue_record,
        approved_record=approved_record,
        trusted_question=trusted,
        expected_reviewer="owner-01",
    )
    generation = payload["generation"]
    if generation["model_id"] != EXPECTED_BASE_ID or generation["model_revision"] != EXPECTED_BASE_REVISION:
        raise RuntimeError("legacy approval is not bound to the pinned base")
    if generation["adapter_id"] != ADAPTER_ID or generation["adapter_revision"] != ADAPTER_REVISION:
        raise RuntimeError("legacy approval is not bound to the pinned adapter")
    approved_mapping = sorted(
        ({"answer": item["answer"], "misconception": item["misconception"]} for item in payload["distractors"]),
        key=lambda item: (item["answer"], item["misconception"]),
    )
    approved_cases.append({
        "kind": "legacy-approved",
        "caseId": question["id"],
        "messages": [
            {"role": "system", "content": LEGACY_SYSTEM_PROMPT},
            {"role": "user", "content": build_legacy_user(question["prompt"], question["correct"], question["topic"])},
        ],
        "maxTokens": 512,
        "correctAnswer": question["correct"],
        "approvedMapping": approved_mapping,
    })
if len(approved_cases) != LEGACY_APPROVED_COUNT:
    raise RuntimeError("the legacy parity set must contain exactly six authenticated approvals")

PARITY_CASES = reference_cases + sorted(approved_cases, key=lambda item: item["caseId"])
if len({case["caseId"] for case in PARITY_CASES}) != REFERENCE_PROMPT_COUNT + LEGACY_APPROVED_COUNT:
    raise RuntimeError("parity corpus contains duplicate identities")
corpus_receipt = {
    "referenceCount": len(reference_cases),
    "approvedLegacyCount": len(approved_cases),
    "casesSha256": sha256_bytes(canonical_json_bytes([{"caseId": case["caseId"], "kind": case["kind"], "messages": case["messages"], "maxTokens": case["maxTokens"]} for case in PARITY_CASES])),
}
atomic_write_json(WORK_ROOT / "parity_corpus_receipt_v1.json", corpus_receipt)
print("Authenticated exactly 60 compiler-replayable prompts plus six owner-approved encounters.")


In [ ]:
import gc
import random
import numpy as np
import torch
from peft import PeftModel
from unsloth import FastLanguageModel

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
if not torch.cuda.is_available():
    raise RuntimeError("select a Colab GPU runtime before loading the model")

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(BASE_SNAPSHOT),
    max_seq_length=CONTEXT_SIZE,
    dtype=None,
    load_in_4bit=True,
    local_files_only=True,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_SNAPSHOT), is_trainable=False, local_files_only=True)
FastLanguageModel.for_inference(model)

def original_completion(messages, max_new_tokens):
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to(next(model.parameters()).device)
    prompt_length = input_ids.shape[-1]
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True, clean_up_tokenization_spaces=False)

ORIGINAL_OUTPUTS_PATH = WORK_ROOT / "original_adapter_outputs_v1.json"
original_outputs = read_json(ORIGINAL_OUTPUTS_PATH) if ORIGINAL_OUTPUTS_PATH.exists() else {}
for case in PARITY_CASES:
    case_id = case["caseId"]
    prompt_sha = sha256_bytes(canonical_json_bytes(case["messages"]))
    existing = original_outputs.get(case_id)
    if existing is not None:
        if existing.get("promptSha256") != prompt_sha or sha256_bytes(existing.get("text", "").encode("utf-8")) != existing.get("textSha256"):
            raise RuntimeError(f"invalid original-output resume record: {case_id}")
        continue
    text = original_completion(case["messages"], case["maxTokens"])
    original_outputs[case_id] = {"promptSha256": prompt_sha, "text": text, "textSha256": sha256_bytes(text.encode("utf-8"))}
    atomic_write_json(ORIGINAL_OUTPUTS_PATH, original_outputs)
if set(original_outputs) != {case["caseId"] for case in PARITY_CASES}:
    raise RuntimeError("original inference did not complete the exact corpus")
print(f"Saved {len(original_outputs)} deterministic original-adapter completions.")


In [ ]:
MERGED_DIR = WORK_ROOT / "merged-16bit"
MERGED_TEMP_DIR = WORK_ROOT / "merged-16bit.partial"
MERGE_RECEIPT_PATH = WORK_ROOT / "merge_receipt_v1.json"
if MERGE_RECEIPT_PATH.exists():
    merge_receipt = read_json(MERGE_RECEIPT_PATH)
    mergedModelTreeSha256, merged_files = hash_tree(MERGED_DIR)
    if merge_receipt.get("mergedModelTreeSha256") != mergedModelTreeSha256:
        raise RuntimeError("merged model resume digest mismatch")
    if merge_receipt.get("baseTreeSha256") != base_tree_sha or merge_receipt.get("adapterTreeSha256") != adapter_tree_sha:
        raise RuntimeError("merged model source receipt mismatch")
else:
    if MERGED_DIR.exists():
        raise RuntimeError("unreceipted merged directory exists; inspect it and choose a clean WORK_ROOT")
    if shutil.disk_usage(WORK_ROOT).free < 9 * 1024**3:
        raise RuntimeError("at least 9 GiB free checkpoint storage is required before the 16-bit merge")
    if MERGED_TEMP_DIR.exists():
        shutil.rmtree(MERGED_TEMP_DIR)
    model.save_pretrained_merged(
        str(MERGED_TEMP_DIR),
        tokenizer,
        save_method="merged_16bit",
    )
    os.replace(MERGED_TEMP_DIR, MERGED_DIR)
    mergedModelTreeSha256, merged_files = hash_tree(MERGED_DIR)
    merge_receipt = {
        "schemaVersion": "wayline.merge-receipt.v1",
        "baseTreeSha256": base_tree_sha,
        "adapterTreeSha256": adapter_tree_sha,
        "mergeMethod": "unsloth.save_pretrained_merged:merged_16bit",
        "mergedModelTreeSha256": mergedModelTreeSha256,
        "files": merged_files,
    }
    atomic_write_json(MERGE_RECEIPT_PATH, merge_receipt)
merged_tokenizer_assets = {item["path"]: item["sha256"] for item in merged_files if Path(item["path"]).name in tokenizer_names}
if merged_tokenizer_assets != tokenizer_assets:
    raise RuntimeError("merged tokenizer or chat-template assets differ from the pinned base")
del model, base_model
gc.collect()
torch.cuda.empty_cache()
print(f"Merged frozen adapter without training: {mergedModelTreeSha256}")


In [ ]:
LLAMA_ROOT = SCRATCH_ROOT / "llama.cpp"
if not LLAMA_ROOT.exists():
    LLAMA_ROOT.mkdir(parents=True)
    run_command(["git", "init", str(LLAMA_ROOT)])
    run_command(["git", "-C", str(LLAMA_ROOT), "remote", "add", "origin", "https://github.com/ggml-org/llama.cpp.git"])
    run_command(["git", "-C", str(LLAMA_ROOT), "fetch", "--depth", "1", "origin", LLAMA_CPP_REVISION])
    run_command(["git", "-C", str(LLAMA_ROOT), "checkout", "--detach", "FETCH_HEAD"])
actual_llama_revision = run_command(["git", "-C", str(LLAMA_ROOT), "rev-parse", "HEAD"], capture_output=True).stdout.strip()
if actual_llama_revision != LLAMA_CPP_REVISION:
    raise RuntimeError("checked-out llama.cpp commit differs from the user-approved pin")

LLAMA_BUILD = LLAMA_ROOT / "build"
QUANTIZER = LLAMA_BUILD / "bin/llama-quantize"
SERVER = LLAMA_BUILD / "bin/llama-server"
if not QUANTIZER.exists() or not SERVER.exists():
    run_command(["cmake", "-S", str(LLAMA_ROOT), "-B", str(LLAMA_BUILD), "-DGGML_CUDA=ON", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"])
    run_command(["cmake", "--build", str(LLAMA_BUILD), "--config", "Release", "--target", "llama-quantize", "llama-server", "-j2"])
CONVERTER = LLAMA_ROOT / "convert_hf_to_gguf.py"
LLAMA_LICENSE = LLAMA_ROOT / "LICENSE"
for required_path in (CONVERTER, QUANTIZER, SERVER, LLAMA_LICENSE):
    if not required_path.is_file():
        raise RuntimeError(f"pinned llama.cpp checkout lacks required artifact: {required_path}")
llama_receipt = {
    "llamaCppRevision": LLAMA_CPP_REVISION,
    "converterSha256": sha256_file(CONVERTER),
    "quantizerSha256": sha256_file(QUANTIZER),
    "serverSha256": sha256_file(SERVER),
    "licenseFilesSha256": {"llama.cpp/LICENSE": sha256_file(LLAMA_LICENSE)},
}
atomic_write_json(WORK_ROOT / "llama_cpp_receipt_v1.json", llama_receipt)
print(f"Built exact llama.cpp commit {LLAMA_CPP_REVISION}.")


In [ ]:
F16_GGUF = SCRATCH_ROOT / "wayline-qwen3-4b-v7-f16.gguf"
F16_GGUF_TEMP = SCRATCH_ROOT / "wayline-qwen3-4b-v7-f16.gguf.partial"
Q4_GGUF_TEMP = SCRATCH_ROOT / (OUTPUT_GGUF_NAME + ".partial")
Q4_GGUF_PERSIST_TEMP = WORK_ROOT / (OUTPUT_GGUF_NAME + ".partial")
Q4_GGUF = WORK_ROOT / OUTPUT_GGUF_NAME
CONVERSION_RECEIPT_PATH = WORK_ROOT / "conversion_receipt_v1.json"
if CONVERSION_RECEIPT_PATH.exists():
    conversion_receipt = read_json(CONVERSION_RECEIPT_PATH)
    ggufF16Sha256 = conversion_receipt.get("ggufF16Sha256")
    ggufSha256 = sha256_file(Q4_GGUF)
    if not isinstance(ggufF16Sha256, str) or re.fullmatch(r"[0-9a-f]{64}", ggufF16Sha256) is None:
        raise RuntimeError("F16 conversion receipt is invalid")
    if conversion_receipt.get("ggufSha256") != ggufSha256 or conversion_receipt.get("mergedModelTreeSha256") != mergedModelTreeSha256 or conversion_receipt.get("quantization") != QUANTIZATION:
        raise RuntimeError("GGUF resume digest mismatch")
else:
    if shutil.disk_usage(SCRATCH_ROOT).free < 12 * 1024**3:
        raise RuntimeError("at least 12 GiB free local scratch is required before conversion")
    if shutil.disk_usage(WORK_ROOT).free < 5 * 1024**3:
        raise RuntimeError("at least 5 GiB free checkpoint storage is required before conversion")
    if Q4_GGUF.exists():
        raise RuntimeError("unreceipted persistent Q4 GGUF exists; inspect it and use a clean WORK_ROOT")
    for disposable_partial in (F16_GGUF, F16_GGUF_TEMP, Q4_GGUF_TEMP, Q4_GGUF_PERSIST_TEMP):
        if disposable_partial.exists():
            disposable_partial.unlink()
    run_command([sys.executable, str(CONVERTER), str(MERGED_DIR), "--outfile", str(F16_GGUF_TEMP), "--outtype", "f16"])
    os.replace(F16_GGUF_TEMP, F16_GGUF)
    run_command([str(QUANTIZER), str(F16_GGUF), str(Q4_GGUF_TEMP), "Q4_K_M"])
    shutil.copy2(Q4_GGUF_TEMP, Q4_GGUF_PERSIST_TEMP)
    os.replace(Q4_GGUF_PERSIST_TEMP, Q4_GGUF)
    ggufF16Sha256 = sha256_file(F16_GGUF)
    ggufSha256 = sha256_file(Q4_GGUF)
    if sha256_file(Q4_GGUF_TEMP) != ggufSha256:
        raise RuntimeError("persistent Q4 copy differs from the quantizer output")
    conversion_receipt = {
        "schemaVersion": "wayline.conversion-receipt.v1",
        "mergedModelTreeSha256": mergedModelTreeSha256,
        "ggufF16Sha256": ggufF16Sha256,
        "ggufSha256": ggufSha256,
        "ggufBytes": Q4_GGUF.stat().st_size,
        "quantization": QUANTIZATION,
        "commandReceiptsSha256": sha256_bytes(canonical_json_bytes(COMMAND_RECEIPTS)),
    }
    atomic_write_json(CONVERSION_RECEIPT_PATH, conversion_receipt)

license_files = {"llama.cpp/LICENSE": sha256_file(LLAMA_LICENSE)}
for label, root in (("base", BASE_SNAPSHOT), ("adapter", ADAPTER_SNAPSHOT)):
    candidates = [path for path in sorted(root.rglob("*")) if path.is_file() and ("license" in path.name.casefold() or path.name.casefold() in {"copying", "notice", "readme.md"})]
    if not candidates:
        raise RuntimeError(f"no license/card source file found for {label}")
    for path in candidates:
        license_files[f"{label}/{path.relative_to(root).as_posix()}"] = sha256_file(path)
licenseFilesSha256 = license_files
export_receipt_draft = {
    "schemaVersion": "wayline.export-receipt.v1",
    "configuration": configuration_receipt,
    "verifiedInputsSha256": sha256_bytes(canonical_json_bytes(EXPECTED_INPUT_SHA256)),
    "baseTreeSha256": base_tree_sha,
    "adapterTreeSha256": adapter_tree_sha,
    "mergedModelTreeSha256": mergedModelTreeSha256,
    "tokenizerAssetSha256": tokenizerAssetSha256,
    "licenseIds": hub_licenses,
    "licenseFilesSha256": licenseFilesSha256,
    "llamaCpp": llama_receipt,
    "commands": COMMAND_RECEIPTS,
    "commandSha256": sha256_bytes(canonical_json_bytes(COMMAND_RECEIPTS)),
    "ggufF16Sha256": ggufF16Sha256,
    "ggufSha256": ggufSha256,
    "ggufBytes": Q4_GGUF.stat().st_size,
    "runtimeVersions": {name: importlib.metadata.version(name) for name in ("unsloth", "torch", "transformers", "peft", "huggingface_hub")},
}
atomic_write_json(WORK_ROOT / "export_receipt_draft_v1.json", export_receipt_draft)
print(f"Converted Q4_K_M GGUF: {ggufSha256} ({Q4_GGUF.stat().st_size:,} bytes). Parity is still required.")


In [ ]:
import socket
import time
from urllib.error import HTTPError, URLError
from urllib.request import ProxyHandler, Request, build_opener

def unused_loopback_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]

port = unused_loopback_port()
loopback_opener = build_opener(ProxyHandler({}))
server_argv = [
    str(SERVER), "--model", str(Q4_GGUF), "--alias", "wayline-qwen",
    "--host", "127.0.0.1", "--port", str(port), "--ctx-size", str(CONTEXT_SIZE),
    "--parallel", "1", "--n-gpu-layers", "99",
]
server_command_receipt = {"argv": server_argv, "commandSha256": sha256_bytes(canonical_json_bytes(server_argv))}
server_process = subprocess.Popen(server_argv, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, text=True)

def json_request(url, payload=None, timeout=300):
    data = None if payload is None else canonical_json_bytes(payload)
    request = Request(url, data=data, headers={"Content-Type": "application/json"}, method="GET" if data is None else "POST")
    with loopback_opener.open(request, timeout=timeout) as response:
        body = response.read(1024 * 1024 + 1)
    if len(body) > 1024 * 1024:
        raise RuntimeError("llama.cpp response exceeded one MiB")
    return json.loads(body, object_pairs_hook=reject_duplicate_keys)

try:
    deadline = time.monotonic() + 180
    while True:
        if server_process.poll() is not None:
            raise RuntimeError("llama-server exited before readiness")
        try:
            json_request(f"http://127.0.0.1:{port}/health", timeout=5)
            break
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError):
            if time.monotonic() >= deadline:
                raise RuntimeError("llama-server readiness deadline expired")
            time.sleep(2)

    GGUF_OUTPUTS_PATH = WORK_ROOT / "gguf_outputs_v1.json"
    gguf_outputs = read_json(GGUF_OUTPUTS_PATH) if GGUF_OUTPUTS_PATH.exists() else {}
    for case in PARITY_CASES:
        case_id = case["caseId"]
        prompt_sha = sha256_bytes(canonical_json_bytes(case["messages"]))
        existing = gguf_outputs.get(case_id)
        if existing is not None:
            if existing.get("promptSha256") != prompt_sha or sha256_bytes(existing.get("text", "").encode("utf-8")) != existing.get("textSha256"):
                raise RuntimeError(f"invalid GGUF-output resume record: {case_id}")
            continue
        payload = {
            "model": "wayline-qwen",
            "messages": case["messages"],
            "temperature": 0,
            "seed": 0,
            "stream": False,
            "max_tokens": case["maxTokens"],
            "chat_template_kwargs": {"enable_thinking": False},
        }
        response = json_request(f"http://127.0.0.1:{port}/v1/chat/completions", payload)
        text = response["choices"][0]["message"]["content"]
        if not isinstance(text, str):
            raise RuntimeError(f"GGUF returned non-text content: {case_id}")
        gguf_outputs[case_id] = {"promptSha256": prompt_sha, "text": text, "textSha256": sha256_bytes(text.encode("utf-8"))}
        atomic_write_json(GGUF_OUTPUTS_PATH, gguf_outputs)
finally:
    server_process.terminate()
    try:
        server_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=20)

if set(gguf_outputs) != {case["caseId"] for case in PARITY_CASES}:
    raise RuntimeError("GGUF inference did not complete the exact corpus")
print(f"Saved {len(gguf_outputs)} deterministic GGUF completions.")


In [ ]:
from services.wayline_forge.app.providers.distractor import PinnedSlmManifest, RawSlmGeneration
from services.wayline_forge.app.safe_numeric import NumericParseError, parse_exact_value
from services.wayline_forge.app.slm_prompt import PROMPT_TEMPLATE_SHA256, build_slm_request

FINAL_ROOT = WORK_ROOT / "production-output"
MANIFEST_PATH = FINAL_ROOT / "model_manifest_v1.json"
if MANIFEST_PATH.exists():
    MANIFEST_PATH.unlink()

def strict_contract(text):
    try:
        value = json.loads(text, object_pairs_hook=reject_duplicate_keys)
    except (json.JSONDecodeError, RuntimeError):
        return None
    if not isinstance(value, dict) or set(value) != {"distractors"} or not isinstance(value["distractors"], list):
        return None
    expected = {"misconception", "computation", "answer"}
    if any(not isinstance(item, dict) or set(item) != expected or not all(isinstance(item[key], str) for key in expected) for item in value["distractors"]):
        return None
    return value["distractors"]

def structural_result(case, text):
    items = strict_contract(text)
    exactly_three = items is not None and len(items) == 3
    if not exactly_three:
        return {"exactlyThree": False, "distinctAnswers": False, "keySafe": False}
    try:
        allow_percent = case["kind"] == "wayline-reference" and case["blueprint"].family_id == "decimal_to_percent"
        answers = [parse_exact_value(item["answer"], allow_percent=allow_percent).value for item in items]
        correct = parse_exact_value(case["correctAnswer"], allow_percent=allow_percent).value
    except NumericParseError:
        return {"exactlyThree": True, "distinctAnswers": False, "keySafe": False}
    return {"exactlyThree": True, "distinctAnswers": len(set(answers)) == 3, "keySafe": correct not in answers}

def product_verifier_result(case, text, artifact_sha, generator_sha):
    manifest = PinnedSlmManifest(
        model_id="wayline-parity-artifact",
        model_sha256=artifact_sha,
        adapter_identity_receipt_sha256=adapter_tree_sha,
        gguf_sha256=artifact_sha,
        generator_identity_receipt_sha256=generator_sha,
        registry_id=registry.registry_id,
        prompt_template_sha256=PROMPT_TEMPLATE_SHA256,
    )
    verifier = DistractorVerifier(compiler, registry, manifest)
    request = build_slm_request(case["blueprint"])
    generation = RawSlmGeneration(
        text=text,
        model_sha256=manifest.model_sha256,
        adapter_identity_receipt_sha256=manifest.adapter_identity_receipt_sha256,
        gguf_sha256=manifest.gguf_sha256,
        generator_identity_receipt_sha256=manifest.generator_identity_receipt_sha256,
        prompt_sha256=request.prompt_sha256,
        prompt_template_sha256=manifest.prompt_template_sha256,
        registry_id=manifest.registry_id,
        generated_at_utc="2026-07-11T00:00:00Z",
    )
    result = verifier.verify_generation(case["blueprint"], generation)
    procedure_ids = [] if not result.accepted else sorted(item.procedure_id for item in result.value.verified_distractors)
    return {"productVerifierPass": result.accepted, "acceptedProcedureIds": procedure_ids, "rejectionCode": result.code}

def approved_mapping(text):
    items = strict_contract(text)
    if items is None or len(items) != 3:
        return None
    return sorted(({"answer": item["answer"], "misconception": item["misconception"]} for item in items), key=lambda item: (item["answer"], item["misconception"]))

original_generator_sha = sha256_bytes(canonical_json_bytes({"runtime": "unsloth", "versions": export_receipt_draft["runtimeVersions"]}))
gguf_generator_sha = sha256_bytes(canonical_json_bytes({"runtime": "llama.cpp", "revision": LLAMA_CPP_REVISION, "serverSha256": llama_receipt["serverSha256"]}))
case_results = []
for case in PARITY_CASES:
    case_id = case["caseId"]
    original_text = original_outputs[case_id]["text"]
    gguf_text = gguf_outputs[case_id]["text"]
    original_result = structural_result(case, original_text)
    gguf_result = structural_result(case, gguf_text)
    if case["kind"] == "wayline-reference":
        original_result.update(product_verifier_result(case, original_text, adapter_tree_sha, original_generator_sha))
        gguf_result.update(product_verifier_result(case, gguf_text, ggufSha256, gguf_generator_sha))
    else:
        original_result["approvedMapping"] = approved_mapping(original_text)
        gguf_result["approvedMapping"] = approved_mapping(gguf_text)
    case_results.append({
        "caseId": case_id,
        "kind": case["kind"],
        "exactOutputAgreement": original_text == gguf_text,
        "original": original_result,
        "gguf": gguf_result,
    })

def rate(side, metric, rows):
    return sum(result[side].get(metric) is True for result in rows) / len(rows)

reference_results = [result for result in case_results if result["kind"] == "wayline-reference"]
metrics = {}
for metric, rows in (("exactlyThree", case_results), ("distinctAnswers", case_results), ("keySafe", case_results), ("productVerifierPass", reference_results)):
    original_rate = rate("original", metric, rows)
    gguf_rate = rate("gguf", metric, rows)
    metrics[metric] = {"original": original_rate, "gguf": gguf_rate, "regression": original_rate - gguf_rate}

gate_errors = []
for metric, values in metrics.items():
    if values["regression"] > MAX_GATE_REGRESSION + 1e-12:
        gate_errors.append(f"{metric} gate regression exceeds 5 percentage points")
for case, result in zip(PARITY_CASES, case_results, strict=True):
    if case["kind"] != "legacy-approved":
        continue
    if result["original"].get("approvedMapping") != case["approvedMapping"] or result["gguf"].get("approvedMapping") != case["approvedMapping"]:
        gate_errors.append(f"{case['caseId']} approved mapping changed")
if gate_errors:
    failed_report = {"schemaVersion": "wayline.parity-report.v1", "passed": False, "errors": gate_errors, "metrics": metrics, "cases": case_results}
    atomic_write_json(WORK_ROOT / "wayline_parity_report_FAILED_v1.json", failed_report)
    raise RuntimeError("parity rejected the GGUF: " + "; ".join(gate_errors))

parity_gate = {
    "schemaVersion": "wayline.parity-report.v1",
    "passed": True,
    "referenceCount": len(reference_results),
    "approvedLegacyCount": len(case_results) - len(reference_results),
    "maximumAllowedRegression": MAX_GATE_REGRESSION,
    "metrics": metrics,
    "exactOutputAgreementRate": sum(result["exactOutputAgreement"] for result in case_results) / len(case_results),
    "cases": case_results,
}
PARITY_REPORT_PATH = WORK_ROOT / "wayline_parity_report_v1.json"
atomic_write_json(PARITY_REPORT_PATH, parity_gate)
print("All mapping and <=5 percentage-point parity gates passed.")


In [ ]:
from services.wayline_forge.app.model_manifest import parse_model_manifest

if parity_gate["passed"] is not True:
    raise RuntimeError("production manifest requires a passing parity receipt")
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = FINAL_ROOT / "model_manifest_v1.json"
model_manifest = {
    "schemaVersion": "wayline.model-manifest.v1",
    "baseModelId": EXPECTED_BASE_ID,
    "baseModelRevision": EXPECTED_BASE_REVISION,
    "adapterId": ADAPTER_ID,
    "adapterRevision": ADAPTER_REVISION,
    "llamaCppRevision": LLAMA_CPP_REVISION,
    "quantization": QUANTIZATION,
    "ggufFileName": OUTPUT_GGUF_NAME,
    "ggufSha256": ggufSha256,
    "promptSha256": PROMPT_TEMPLATE_SHA256,
    "tokenizerSha256": tokenizerAssetSha256,
    "contextSize": CONTEXT_SIZE,
    "threadCount": MAC_THREAD_COUNT,
    "platform": "macos-arm64",
}
manifest_bytes = canonical_json_bytes(model_manifest) + b"\n"
parse_model_manifest(manifest_bytes)
MANIFEST_PATH.write_bytes(manifest_bytes)

export_receipt = dict(export_receipt_draft)
export_receipt.update({
    "servingCommand": server_command_receipt,
    "parityReportSha256": sha256_file(PARITY_REPORT_PATH),
    "modelManifestSha256": sha256_file(MANIFEST_PATH),
    "promptSha256": PROMPT_TEMPLATE_SHA256,
})
EXPORT_RECEIPT_PATH = FINAL_ROOT / "wayline_export_receipt_v1.json"
atomic_write_json(EXPORT_RECEIPT_PATH, export_receipt)
FINAL_PARITY_PATH = FINAL_ROOT / "wayline_parity_report_v1.json"
shutil.copy2(PARITY_REPORT_PATH, FINAL_PARITY_PATH)

NOTICE_PATH = FINAL_ROOT / "THIRD_PARTY_NOTICES.md"
NOTICE_PATH.write_text(
    "# Third-party source receipt\n\n"
    f"- Base: `{EXPECTED_BASE_ID}` at `{EXPECTED_BASE_REVISION}`; declared license `{hub_licenses['base']}`.\n"
    f"- Adapter: `{ADAPTER_ID}` at `{ADAPTER_REVISION}`; declared license `{hub_licenses['adapter']}`.\n"
    f"- llama.cpp: `ggml-org/llama.cpp` at `{LLAMA_CPP_REVISION}`; license file SHA-256 `{sha256_file(LLAMA_LICENSE)}`.\n\n"
    "These receipts are not legal advice. Review the included source notices before redistribution.\n",
    encoding="utf-8",
)
print(f"Production manifest emitted only after parity: {MANIFEST_PATH}")


In [ ]:
PACKAGE_ROOT = SCRATCH_ROOT / "wayline_live_forge_q4_k_m"
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
PACKAGE_ROOT.mkdir(parents=True)
for source in (Q4_GGUF, MANIFEST_PATH, EXPORT_RECEIPT_PATH, FINAL_PARITY_PATH, NOTICE_PATH):
    shutil.copy2(source, PACKAGE_ROOT / source.name)
LICENSE_OUTPUT = PACKAGE_ROOT / "licenses"
for label, root in (("base", BASE_SNAPSHOT), ("adapter", ADAPTER_SNAPSHOT)):
    for path in sorted(root.rglob("*")):
        if path.is_file() and ("license" in path.name.casefold() or path.name.casefold() in {"copying", "notice", "readme.md"}):
            destination = LICENSE_OUTPUT / label / path.relative_to(root)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, destination)
llama_license_destination = LICENSE_OUTPUT / "llama.cpp/LICENSE"
llama_license_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(LLAMA_LICENSE, llama_license_destination)

package_files = [path for path in sorted(PACKAGE_ROOT.rglob("*")) if path.is_file()]
sha_lines = [f"{sha256_file(path)}  {path.relative_to(PACKAGE_ROOT).as_posix()}" for path in package_files]
SHA256SUMS_PATH = PACKAGE_ROOT / "SHA256SUMS"
SHA256SUMS_PATH.write_text("\n".join(sha_lines) + "\n", encoding="utf-8")
PACKAGE_ZIP = FINAL_ROOT / "wayline_live_forge_q4_k_m.zip"
with zipfile.ZipFile(PACKAGE_ZIP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(PACKAGE_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, (Path(PACKAGE_ROOT.name) / path.relative_to(PACKAGE_ROOT)).as_posix())
package_receipt = {"zipSha256": sha256_file(PACKAGE_ZIP), "zipBytes": PACKAGE_ZIP.stat().st_size, "ggufSha256": ggufSha256}
PACKAGE_RECEIPT_PATH = FINAL_ROOT / "package_receipt_v1.json"
atomic_write_json(PACKAGE_RECEIPT_PATH, package_receipt)
files.download(str(MANIFEST_PATH))
files.download(str(FINAL_PARITY_PATH))
files.download(str(PACKAGE_RECEIPT_PATH))
files.download(str(PACKAGE_ZIP))
print(f"Packaged downloadable artifacts: {PACKAGE_ZIP}")
